# Model-based reinforcement learning: lab notebook

**Machine Learning (5CS4-05), Semi-supervised and reinforcement learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np

### 1. The Dyna maze (6 x 9). START at (2,0), GOAL at (0,8). Reward 1 only at the goal.

In [ ]:
# 1. The Dyna maze (6 x 9). START at (2,0), GOAL at (0,8). Reward 1 only at the goal.
rows = [".......#G", "..#....#.", "S.#....#.", "..#......", ".....#...", "........."]
R, C = len(rows), len(rows[0])
start = 2 * C
goal = 8
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

def step(s, a):
    r, c = s // C + ACTIONS[a][0], s % C + ACTIONS[a][1]
    s2 = s if not (0 <= r < R and 0 <= c < C) or rows[r][c] == "#" else r * C + c
    return s2, (1.0 if s2 == goal else 0.0), s2 == goal

### 2. Dyna-Q: direct learning + model learning + n planning updates per real step

In [ ]:
# 2. Dyna-Q: direct learning + model learning + n planning updates per real step
def dyna_q(n_planning, episodes=30, alpha=0.1, gamma=0.95, eps=0.1, seed=1):
    rng = np.random.default_rng(seed)
    Q = np.zeros((R * C, 4)); model = {}; steps_per_episode = []
    for _ in range(episodes):
        s, t = start, 0
        while True:
            a = int(rng.integers(4)) if rng.random() < eps else int(rng.choice(np.flatnonzero(Q[s] >= Q[s].max() - 1e-12)))
            s2, r, done = step(s, a); t += 1
            Q[s, a] += alpha * (r + (0 if done else gamma * Q[s2].max()) - Q[s, a])     # direct RL
            model[(s, a)] = (r, s2, done)                                               # model learning
            keys = list(model.keys())
            for _ in range(n_planning):                                                  # planning
                ps, pa = keys[rng.integers(len(keys))]
                pr, ps2, pdone = model[(ps, pa)]
                Q[ps, pa] += alpha * (pr + (0 if pdone else gamma * Q[ps2].max()) - Q[ps, pa])
            s = s2
            if done:
                break
        steps_per_episode.append(t)
    return steps_per_episode, len(model)

### 3. Compare different amounts of planning

In [ ]:
# 3. Compare different amounts of planning
for n in (0, 5, 50):
    steps, size = dyna_q(n)
    print(f"n = {n:2d}: steps in episodes 1, 2, 3, 5, 10, 30 = {[steps[i - 1] for i in (1, 2, 3, 5, 10, 30)]}   model size = {size}")

### 4. Counting updates

In [ ]:
# 4. Counting updates
T, n = 100, 10
print("Q updates after", T, "real steps with n =", n, ":", T * (1 + n))

## Expected output

Running all cells above should print:

```
n =  0: steps in episodes 1, 2, 3, 5, 10, 30 = [228, 1075, 363, 384, 138, 14]   model size = 183
n =  5: steps in episodes 1, 2, 3, 5, 10, 30 = [858, 245, 16, 14, 18, 20]   model size = 169
n = 50: steps in episodes 1, 2, 3, 5, 10, 30 = [2083, 44, 16, 15, 18, 14]   model size = 181
Q updates after 100 real steps with n = 10 : 1100
```

## Practice exercises

Try these in the cells below.

1. Run Dyna-Q with n = 0, 5, 50 and print the steps taken in the 10th episode for each.
2. Change the maze after 20 episodes (open a shortcut). Does Dyna-Q find it?
3. Replace the deterministic model with count-based estimates for a slippery grid.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
